# Part 3 Transformer-Based Sentiment Analysis
Welcome to the state-of-the-art! In this notebook, we abandon traditional counting/statistics and sequence-level RNNs, and step into the world of **Transformers** using Hugging Face's `transformers` library.

We will:
1. Load a pretrained transformer (**BERT**).
2. Tokenize our text specifically for BERT.
3. Fine-tune BERT on our movie review dataset using the `Trainer` API.
4. Evaluate its performance and context understanding.

In [5]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Ensure GPU is used if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# We take a SMALL SUBSET of the data for this notebook. 
# Fine-tuning BERT on all 25,000 reviews takes several hours on a standard machine.
print("Loading a subset of the dataset for speed...")
train_df = pd.read_csv('../data/imdb_train.csv').sample(5000, random_state=42)
test_df = pd.read_csv('../data/imdb_test.csv').sample(1500, random_state=42)

train_df['label'] = train_df['sentiment'].map({"Negative": 0, "Positive": 1})
test_df['label'] = test_df['sentiment'].map({"Negative": 0, "Positive": 1})

# Convert Pandas DataFrames into Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['text', 'label']])

Using device: cuda
Loading a subset of the dataset for speed...


## Step 1 & 2 Load Pretrained Tokenizer & Model
BERT understands text completely differently than TF-IDF. It breaks words into "sub-words" (e.g., "playing" -> "play", "##ing") and maps them to highly contextual mathematical vectors.

We use `bert-base-uncased` which means all text is converted to lowercase.

In [6]:
model_name = "bert-base-uncased"

print(f"Loading tokenizer: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model architecture...")
# We specify num_labels=2 because we have Positive (1) and Negative (0)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

def tokenize_function(examples):
    # Padding and truncation ensure all inputs are the exact same length (512 tokens max for BERT)
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=256)

print("Tokenizing training data...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)

print("Tokenizing testing data...")
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Loading tokenizer: bert-base-uncased
Loading model architecture...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenizing training data...


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing testing data...


Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [10]:
model2_name = "roberta-base"

print(f"Loading tokenizer: {model2_name}")
tokenizer2 = AutoTokenizer.from_pretrained(model2_name)

print("Loading model architecture...")
model2 = AutoModelForSequenceClassification.from_pretrained(model2_name, num_labels=2)
model2.to(device)

def tokenize2_function(examples):
    # CRITICAL CHANGE: RoBERTa does not use 'token_type_ids'. 
    # We must explicitly pop or exclude it if your dataset contains it,
    # or ensure our tokenizer doesn't return it.
    return tokenizer(
        examples['text'], 
        padding="max_length", 
        truncation=True, 
        max_length=256,
        return_token_type_ids=False # <-- ADD THIS LINE
    )

print("Tokenizing training data...")
tokenized2_train = train_dataset.map(tokenize2_function, batched=True)

print("Tokenizing testing data...")
tokenized2_test = test_dataset.map(tokenize2_function, batched=True)

Loading tokenizer: roberta-base


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

c:\Users\padol\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\padol\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\padol\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and 

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenizing training data...


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing testing data...


Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

## Step 3 Fine-Tuning with Trainer API
Hugging Face provides an incredible class called `Trainer` that handles all the complex PyTorch training loops for us.
We just need to define how we want it to evaluate (e.g., compute accuracy) and define our training hyperparameters.

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Define training arguments
training_args = TrainingArguments(
    output_dir='../models/bert_sentiment',
    num_train_epochs=2,              # Train for 2 passes over the data
    per_device_train_batch_size=8,   # Small batch size to avoid running out of memory
    per_device_eval_batch_size=16,
    evaluation_strategy="epoch",     # Evaluate at the end of each epoch
    save_strategy="epoch",
    logging_dir='./logs',
    learning_rate=1e-5,              # Tiny learning rate because BERT is already pre-trained
    weight_decay=0.1,
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)

print("Starting Fine-Tuning (This will take time depending on your hardware!)...")
trainer.train()

c:\Users\padol\anaconda3\Lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Starting Fine-Tuning (This will take time depending on your hardware!)...


  0%|          | 0/1250 [00:00<?, ?it/s]

{'loss': 0.3822, 'grad_norm': 0.6627237200737, 'learning_rate': 6e-06, 'epoch': 0.8}


  0%|          | 0/94 [00:00<?, ?it/s]

{'eval_loss': 0.30206000804901123, 'eval_accuracy': 0.9013333333333333, 'eval_f1': 0.9019867549668874, 'eval_precision': 0.8719590268886044, 'eval_recall': 0.934156378600823, 'eval_runtime': 13.2451, 'eval_samples_per_second': 113.25, 'eval_steps_per_second': 7.097, 'epoch': 1.0}
{'loss': 0.2492, 'grad_norm': 33.083213806152344, 'learning_rate': 2.0000000000000003e-06, 'epoch': 1.6}


  0%|          | 0/94 [00:00<?, ?it/s]

{'eval_loss': 0.3512813448905945, 'eval_accuracy': 0.9073333333333333, 'eval_f1': 0.9068988613529806, 'eval_precision': 0.8861256544502618, 'eval_recall': 0.9286694101508917, 'eval_runtime': 13.2228, 'eval_samples_per_second': 113.441, 'eval_steps_per_second': 7.109, 'epoch': 2.0}
{'train_runtime': 384.195, 'train_samples_per_second': 26.028, 'train_steps_per_second': 3.254, 'train_loss': 0.2901220703125, 'epoch': 2.0}


TrainOutput(global_step=1250, training_loss=0.2901220703125, metrics={'train_runtime': 384.195, 'train_samples_per_second': 26.028, 'train_steps_per_second': 3.254, 'total_flos': 1315555276800000.0, 'train_loss': 0.2901220703125, 'epoch': 2.0})

In [11]:

# Define training arguments
training2_args = TrainingArguments(
    output_dir='../models/roberta_sentiment',
    num_train_epochs=2,              # Train for 2 passes over the data
    per_device_train_batch_size=8,   # Small batch size to avoid running out of memory
    per_device_eval_batch_size=16,
    evaluation_strategy="epoch",     # Evaluate at the end of each epoch
    save_strategy="epoch",
    logging_dir='./logs',
    learning_rate=1e-5,              # Tiny learning rate because BERT is already pre-trained
    weight_decay=0.1,
)

# Initialize the Trainer
trainer2 = Trainer(
    model=model2,
    args=training2_args,
    train_dataset=tokenized2_train,
    eval_dataset=tokenized2_test,
    compute_metrics=compute_metrics
)

print("Starting Fine-Tuning (This will take time depending on your hardware!)...")
trainer2.train()

c:\Users\padol\anaconda3\Lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Starting Fine-Tuning (This will take time depending on your hardware!)...


  0%|          | 0/1250 [00:00<?, ?it/s]

{'loss': 0.6977, 'grad_norm': 2.0490739345550537, 'learning_rate': 6e-06, 'epoch': 0.8}


  0%|          | 0/94 [00:00<?, ?it/s]

{'eval_loss': 0.6904446482658386, 'eval_accuracy': 0.5353333333333333, 'eval_f1': 0.16924910607866508, 'eval_precision': 0.6454545454545455, 'eval_recall': 0.09739368998628258, 'eval_runtime': 15.3035, 'eval_samples_per_second': 98.017, 'eval_steps_per_second': 6.142, 'epoch': 1.0}
{'loss': 0.6871, 'grad_norm': 7.029976844787598, 'learning_rate': 2.0000000000000003e-06, 'epoch': 1.6}


  0%|          | 0/94 [00:00<?, ?it/s]

{'eval_loss': 0.6212115287780762, 'eval_accuracy': 0.6653333333333333, 'eval_f1': 0.6714659685863874, 'eval_precision': 0.6420525657071339, 'eval_recall': 0.7037037037037037, 'eval_runtime': 15.3041, 'eval_samples_per_second': 98.013, 'eval_steps_per_second': 6.142, 'epoch': 2.0}
{'train_runtime': 422.3469, 'train_samples_per_second': 23.677, 'train_steps_per_second': 2.96, 'train_loss': 0.6816076049804688, 'epoch': 2.0}


TrainOutput(global_step=1250, training_loss=0.6816076049804688, metrics={'train_runtime': 422.3469, 'train_samples_per_second': 23.677, 'train_steps_per_second': 2.96, 'total_flos': 1315555276800000.0, 'train_loss': 0.6816076049804688, 'epoch': 2.0})

## Step 4 Evaluate Results
Now we see how well BERT performs on the test set.

In [8]:
print("Evaluating on test set...")
results = trainer.evaluate()
print("\n--- BERT Results ---")
for key, value in results.items():
    print(f"{key}: {value}")

# Save the final model
trainer.save_model('../models/bert_sentiment_final')
tokenizer.save_pretrained('../models/bert_sentiment_final')
print("Model saved successfully!")

Evaluating on test set...


  0%|          | 0/94 [00:00<?, ?it/s]


--- BERT Results ---
eval_loss: 0.3512813448905945
eval_accuracy: 0.9073333333333333
eval_f1: 0.9068988613529806
eval_precision: 0.8861256544502618
eval_recall: 0.9286694101508917
eval_runtime: 15.6748
eval_samples_per_second: 95.695
eval_steps_per_second: 5.997
epoch: 2.0
Model saved successfully!


In [12]:
print("Evaluating on test set...")
results2 = trainer2.evaluate()
print("\n--- BERT Results ---")
for key, value in results2.items():
    print(f"{key}: {value}")

# Save the final model
trainer2.save_model('../models/roberta_sentiment_final')
tokenizer2.save_pretrained('../models/roberta_sentiment_final')
print("Model saved successfully!")

Evaluating on test set...


  0%|          | 0/94 [00:00<?, ?it/s]


--- BERT Results ---
eval_loss: 0.6212115287780762
eval_accuracy: 0.6653333333333333
eval_f1: 0.6714659685863874
eval_precision: 0.6420525657071339
eval_recall: 0.7037037037037037
eval_runtime: 17.5184
eval_samples_per_second: 85.624
eval_steps_per_second: 5.366
epoch: 2.0
Model saved successfully!


## Context Understanding & Sarcasm (Bonus Comparison)
Traditional models struggle with sarcasm because they only look at individual words. BERT looks at the entire sentence context bidirectionally. Let's test it on a tricky sentence!

In [14]:
from transformers import pipeline

# Load our newly trained model into an easy-to-use pipeline
sentiment_pipeline = pipeline("sentiment-analysis", model='../models/bert_sentiment_final', tokenizer='../models/bert_sentiment_final')
sentiment2_pipeline = pipeline("sentiment-analysis", model='../models/roberta_sentiment_final', tokenizer='../models/roberta_sentiment_final')

tricky_sentences = [
    "I absolutely loved wasting two hours of my life on this garbage.", # Sarcasm
    "The movie was not terrible, I actually quite liked it.",          # Negation
    "A masterpiece of terrible writing."                               # Contradictory
]

print("--- BERT Inference on Tricky Sentences ---")
for sentence in tricky_sentences:
    result = sentiment_pipeline(sentence)[0]
    result2 = sentiment2_pipeline(sentence)[0]
    # label_1 usually means positive in our mapping, label_0 means negative.
    # The pipeline outputs LABEL_0 or LABEL_1.
    label_name = "Positive" if result['label'] == 'LABEL_1' else "Negative"
    label_name2 = "Positive" if result2['label'] == 'LABEL_1' else "Negative"
    print(f"Sentence: '{sentence}'")
    print(f"Prediction: {label_name} (Confidence: {result['score']:.4f})\n")
    print(f"Prediction 2: {label_name2} (Confidence: {result2['score']:.4f})\n")

--- BERT Inference on Tricky Sentences ---
Sentence: 'I absolutely loved wasting two hours of my life on this garbage.'
Prediction: Negative (Confidence: 0.9703)

Prediction 2: Positive (Confidence: 0.5325)

Sentence: 'The movie was not terrible, I actually quite liked it.'
Prediction: Positive (Confidence: 0.9088)

Prediction 2: Positive (Confidence: 0.5273)

Sentence: 'A masterpiece of terrible writing.'
Prediction: Negative (Confidence: 0.9935)

Prediction 2: Positive (Confidence: 0.5287)

